# Backtest สำหรับเปเปอร์ — ขั้น 3.1 + 3.2notebook นี้ทำสองอย่างต่อกันในที่เดียว| ขั้น | ทำอะไร | เวลา ||---|---|---|| **3.1** | build stack ทั้งคลังจากภาพดิบ (ผ่าน despeckle แล้ว) | ~40 นาที || **3.2** | backtest → ตาราง + รูปที่ใส่เปเปอร์ได้เลย | ~10 นาที |## ทำไมต้องรวมกัน ไม่แยกรัน`.npz` จากขั้น 3.1 **มีประโยชน์อย่างเดียวคือเป็น input ของ 3.2** ถ้าแยกรันคนละที่ต้องขนไฟล์ไปมา และเสี่ยงหยิบ stack ผิดรุ่น (ปัญหาเดิมที่เราเพิ่งแก้ไป)อยู่ใน session เดียวกันแล้วตัวเลขที่ออกมาผูกกับ stack ชุดที่เพิ่ง build เสมอ## ทำไมต้องรัน backtest ใหม่ตัวเลขชุดเก่า (502 origins · CSI 0.465 → 0.098) รันตอนที่ระบบยังเป็นคนละเวอร์ชัน| | ตอนรันเก่า | ตอนนี้ ||---|---|---|| motion engine | `light` block matching | **`pysteps`** optical flow || stack ผ่าน despeckle | ❌ | ✅ || ขนาดคลัง | 945 เฟรม | 1,092+ เฟรม |ถ้าไม่รันใหม่ ตัวเลขในเปเปอร์จะอธิบายระบบคนละตัวกับที่เสนอ และ reviewer ที่ลองreproduce จะได้ผลไม่ตรง> **ค่าที่คาดไว้:** ควรดีขึ้นราว 0.5–2% เท่านั้น ถ้าเปลี่ยนเยอะกว่านั้นมาก> **ให้หยุดหาสาเหตุ อย่าเพิ่งดีใจ** — แปลว่ามีบางอย่างที่เรายังไม่เข้าใจ---

## 3.1 — เตรียมและ build stack

In [ ]:
# build_stack ไม่ต้องใช้ tesseract (เวลามาจากชื่อไฟล์ ไม่ใช่ OCR)# ต้องการแค่ numpy/scipy/Pillow/PyYAML ซึ่ง Colab มีให้อยู่แล้ว# pysteps + opencv ต้องลงเพิ่มสำหรับขั้น 3.2!pip install -q pysteps opencv-python-headlessimport importlib.metadata as md_from pysteps.motion.lucaskanade import dense_lucaskanadeprint("pysteps", md_.version("pysteps"), "พร้อม")

In [ ]:
import os, sys, time, json, subprocessfrom pathlib import PathREPO = "https://github.com/jamorn12/tmd-radar-archive.git"ROOT = Path("/content/tmd-radar-archive")if ROOT.exists():    print("มีอยู่แล้ว — ดึงของใหม่")    !cd {ROOT} && git pull -qelse:    !git clone --depth 1 -q {REPO} {ROOT}os.chdir(ROOT); sys.path.insert(0, str(ROOT))COMMIT = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()print("commit:", COMMIT)

### ด่านตรวจ — อย่าข้ามเช็คว่า clone ที่ได้มามี despeckle อยู่ใน `build_stack` จริง ถ้าไม่มีแล้วรันต่อไปจะเสียเวลา 40 นาทีไปกับ stack รุ่นเก่า แล้วได้ตัวเลขที่เอาไปใช้ไม่ได้

In [ ]:
import inspectfrom radar_archive import build_stack, nowcast, qc, gridfrom radar_archive.config import get_station, CONFIG_PATHsig = inspect.signature(build_stack.build_run)assert "despeckle" in sig.parameters, "❌ build_run ยังไม่มี despeckle — clone เป็นรุ่นเก่า"assert sig.parameters["despeckle"].default is True, "❌ despeckle ไม่ได้เปิดเป็นค่าเริ่มต้น"assert hasattr(qc, "despeckle"), "❌ qc.despeckle ไม่มี"assert qc.despeckle is nowcast.despeckle, "❌ qc กับ nowcast ใช้คนละฟังก์ชัน"print("✅ ด่านตรวจผ่าน — build_run signature:", sig)

### เลือกที่เก็บ `.npz`stack ทั้งคลังรวมกันราว **6 MB** เท่านั้น (บีบอัดแล้ว ส่วนใหญ่เป็นศูนย์)ต่อ Google Drive ไว้จะดีกว่า เพราะถ้า Colab ตัดการเชื่อมต่อระหว่างทางจะไม่เสีย 40 นาทีไปเปล่า ๆ — รันใหม่แล้วมันข้ามไฟล์ที่ build ไว้แล้ว

In [ ]:
USE_DRIVE = True          # False = เก็บใน /content (หายเมื่อ session จบ)if USE_DRIVE:    from google.colab import drive    drive.mount("/content/drive")    STACK_DIR = Path("/content/drive/MyDrive/tmd_radar/stacks")else:    STACK_DIR = Path("/content/stacks")STACK_DIR.mkdir(parents=True, exist_ok=True)print("เก็บ stack ที่:", STACK_DIR)

### buildราว **2 วินาทีต่อเฟรม** · ไฟล์ที่มีอยู่แล้วจะถูกข้าม รันซ้ำได้ปลอดภัย`MIN_RUN = 12` คือช่วงต่อเนื่องต้องยาวอย่างน้อย 12 เฟรม (3 ชม.) — ต้องมี 4 เฟรมเป็น input และอีก 8 เฟรมเป็นของจริงไว้เทียบที่ T+120 ช่วงที่สั้นกว่านี้ทำ backtest ไม่ได้

In [ ]:
STATION = "PHS"MIN_RUN = 12st = get_station(STATION, str(CONFIG_PATH))frames = build_stack.find_frames(Path("data"), st.code)runs = [r for r in build_stack.split_runs(frames) if len(r) >= MIN_RUN]print(f"ไฟล์ดิบ {len(frames)} เฟรม · ช่วงที่ใช้ได้ {len(runs)} ช่วง "      f"({sum(len(r) for r in runs)} เฟรม)")for r in runs:    print(f"  {r[0][0]:%d %b %H:%M}-{r[-1][0]:%H:%M}Z  {len(r):3d} เฟรม")t0, paths, n_spk_total = time.time(), [], 0for r in runs:    p = STACK_DIR / f"{st.code}_{build_stack.run_name(r)}_mean.npz"    if p.exists():        print(f"[cache] {p.name}")    else:        stack, times, meta, rows = build_stack.build_run(r, st, Path("data"), verbose=False)        n = sum(x["despeckled"] for x in rows)        n_spk_total += n        grid.save_stack(p, stack, times, meta)        print(f"[build] {p.name}  {stack.shape}  despeckle -{n}")    paths.append(p)print(f"\nเสร็จใน {(time.time()-t0)/60:.1f} นาที · ตัด speckle รวม {n_spk_total} เซลล์")

---## 3.2 — backtest### วิธีนับแต่ละ origin `i` ใช้ `stack[i-3..i]` หา motion แล้วพยากรณ์ไปข้างหน้าเทียบกับ `stack[i+1..i+8]` ที่สังเกตได้จริงสะสม hit / false alarm / miss **รายอัน** ไว้ก่อน แล้วค่อยรวมทีเดียว (*pooled*)ไม่ใช่เฉลี่ย CSI รายอันแล้วเฉลี่ยอีกที — origin ที่ฝนน้อยจะมี CSI แกว่งสุดขั้วแล้วถ่วงค่าเฉลี่ยผิดสัดส่วนเก็บรายอันไว้ยังทำให้คำนวณ **bootstrap CI** ได้จากข้อมูลชุดเดิม ไม่ต้องรันซ้ำ### metric ที่รายงาน| | สูตร | บอกอะไร ||---|---|---|| CSI | H/(H+FA+M) | ภาพรวม — ตัวหลัก || POD | H/(H+M) | จับฝนที่เกิดจริงได้กี่ % || FAR | FA/(H+FA) | ที่เตือนไป ผิดกี่ % || BIAS | (H+FA)/(H+M) | >1 = พยากรณ์ฝนมากเกินจริง |**ต้องรายงาน FAR ควบคู่ CSI เสมอ** โดยเฉพาะที่ lead ยาว — CSI 0.10 ที่ T+120ฟังดูแค่ต่ำ แต่ FAR 0.82 แปลว่าเตือน 5 ครั้งผิด 4 ซึ่งเป็นคนละเรื่องในทางปฏิบัติ

In [ ]:
import numpy as np, warningswarnings.filterwarnings("ignore", message=".*Singular matrix.*")ENGINES = ("pysteps", "light")   # pysteps = ที่ใช้จริงในระบบ · light = fallbackLEADS   = nowcast.LEADS_MINN_IN    = nowcast.N_INPUTTHR     = 11.98                  # dBZ = 0.1 มม./ชม. (Rosenfeld tropical)def contingency(fc, ob, thr=THR):    ok = np.isfinite(fc) & np.isfinite(ob)    f, o = fc[ok] >= thr, ob[ok] >= thr    return np.array([(f & o).sum(), (f & ~o).sum(), (~f & o).sum()], np.int64)def metrics(c):    h, fa, m = float(c[0]), float(c[1]), float(c[2])    return dict(        CSI  = h/(h+fa+m) if (h+fa+m) else np.nan,        POD  = h/(h+m)    if (h+m)    else np.nan,        FAR  = fa/(h+fa)  if (h+fa)   else np.nan,        BIAS = (h+fa)/(h+m) if (h+m)  else np.nan,    )print("threshold", THR, "dBZ · leads", LEADS, "นาที ·", N_IN, "เฟรม input")

### วนทุก origin — รอบเดียวจบถ้า `estimate_motion` ตกกลับไปใช้อีก engine จะนับเป็น fail ไม่เอามารวมไม่งั้นตัวเลขสอง engine จะปนกันโดยไม่รู้ตัว

In [ ]:
SERIES = list(ENGINES) + ["persist"]CT = {e: {l: [] for l in LEADS} for e in SERIES}   # contingency รายอันINFO = []                                          # metadata รายอันt0 = 0.0import time as _t; t0 = _t.time()for p in paths:    dbz, times = grid.load_stack(p)    kpp = float(np.load(p)["kmperpixel"]); ts = 15.0    steps = [int(round(l/ts)) for l in LEADS]    n_ok = 0    for i in range(N_IN-1, len(dbz)-max(steps)):        inp = dbz[i-N_IN+1 : i+1]        truth = {l: dbz[i+s] for l, s in zip(LEADS, steps)}        row = {"run": p.name, "t0": times[i].isoformat()}        got = {}        for e in ENGINES:            try:                V, minfo = nowcast.estimate_motion(inp, e, kpp, ts)                if minfo["engine"] != e:                    raise RuntimeError(f"ตกไปใช้ {minfo['engine']}")                fc, _ = nowcast.run_extrapolation(inp[-1], V, e, LEADS, ts)                got[e] = fc                s = nowcast.motion_stability(V, minfo, kpp, ts)                row[e] = dict(kmh=s.get("kmh"), bearing=s.get("bearing"),                              conf=s.get("confidence"))            except Exception as ex:                row[e] = dict(error=f"{type(ex).__name__}: {ex}")        if len(got) != len(ENGINES):            continue                                  # ข้าม origin ที่ไม่ครบทุก engine        for e in ENGINES:            for l, f in zip(LEADS, got[e]):                CT[e][l].append(contingency(f, truth[l]))        for l in LEADS:            CT["persist"][l].append(contingency(dbz[i], truth[l]))        fin = np.isfinite(dbz[i])        row["wet_pct"] = round(float((dbz[i][fin] >= THR).mean()*100), 3)        INFO.append(row); n_ok += 1    print(f"{p.name}  {n_ok:4d} origins")N = len(INFO)dt = _t.time()-t0print(f"\nรวม {N} origins · {dt/60:.1f} นาที ({dt/max(N,1):.2f} s/origin)")

### ตารางที่ 1 — forecast skill (เอาลงเปเปอร์ได้)

In [ ]:
import pandas as pdARR = {e: {l: np.array(CT[e][l]) for l in LEADS} for e in SERIES}rows = []for l in LEADS:    r = {"lead_min": l}    for e in SERIES:        m = metrics(ARR[e][l].sum(0))        tag = {"pysteps": "ps", "light": "lt", "persist": "pe"}[e]        for k, v in m.items():            if e == "persist" and k != "CSI":                continue            r[f"{k}_{tag}"] = round(v, 4)    r["skill"] = round(r["CSI_ps"] - r["CSI_pe"], 4)      # เหนือ persistence    r["dCSI"]  = round(r["CSI_ps"] - r["CSI_lt"], 4)      # pysteps - light    rows.append(r)tab = pd.DataFrame(rows)tab

### ความไม่แน่นอน — bootstrapสุ่ม origin คืนที่ 1,000 รอบจาก contingency ที่เก็บไว้แล้ว ไม่ต้องรันพยากรณ์ใหม่⚠️ **origin ในช่วงต่อเนื่องเดียวกันไม่เป็นอิสระต่อกัน** (ฝนก้อนเดิมโผล่หลาย origin)การสุ่มทีละ origin จึงให้ CI **แคบเกินจริง** จึงทำ **block bootstrap** ด้วย —สุ่มทีละช่วงต่อเนื่อง ซึ่งเข้มงวดกว่าและเป็นตัวที่ควรอ้างในเปเปอร์

In [ ]:
def boot_ci(series, lead, n_boot=1000, block=False, seed=0):    A = ARR[series][lead]    rng = np.random.default_rng(seed)    if block:        runs_of = np.array([r["run"] for r in INFO])        uniq = np.unique(runs_of)        idx_by_run = [np.flatnonzero(runs_of == u) for u in uniq]        if len(idx_by_run) < 5:            # สุ่มช่วงคืนที่จากช่วงไม่กี่ช่วง -> ได้ชุดเดิมแทบทุกรอบ CI จะแคบจนเป็นศูนย์            # ไม่ใช่ว่าแม่นยำ แต่เพราะไม่มีความหลากหลายให้สุ่ม            return np.array([np.nan, np.nan])        vals = []        for _ in range(n_boot):            pick = rng.integers(0, len(idx_by_run), len(idx_by_run))            idx = np.concatenate([idx_by_run[k] for k in pick])            vals.append(metrics(A[idx].sum(0))["CSI"])    else:        n = len(A)        vals = [metrics(A[rng.integers(0, n, n)].sum(0))["CSI"] for _ in range(n_boot)]    return np.percentile(vals, [2.5, 97.5])ci = []for l in LEADS:    lo1, hi1 = boot_ci("pysteps", l, block=False)    lo2, hi2 = boot_ci("pysteps", l, block=True)    ci.append(dict(lead_min=l,                   CSI=round(metrics(ARR["pysteps"][l].sum(0))["CSI"], 4),                   ci_lo=round(lo1, 4), ci_hi=round(hi1, 4),                   block_lo=round(lo2, 4), block_hi=round(hi2, 4)))cidf = pd.DataFrame(ci)cidf

### skill horizonนิยามที่ใช้: lead ที่ยาวที่สุดซึ่ง **CSI ยังชนะ persistence อย่างมีนัยสำคัญ**(ช่วง 95% แบบ block ของ `CSI_pysteps − CSI_persist` ยังไม่คร่อม 0)นิยามนี้เข้มกว่าการดู CSI เฉย ๆ เพราะถ้าชนะ persistence ไม่ได้ระบบก็ไม่ได้เพิ่มอะไรจากการเดาว่าฝนไม่ขยับ

In [ ]:
runs_of = np.array([r["run"] for r in INFO])uniq = np.unique(runs_of)idx_by_run = [np.flatnonzero(runs_of == u) for u in uniq]rng = np.random.default_rng(1)N_RUNS = len(idx_by_run)RELIABLE = N_RUNS >= 5if not RELIABLE:    print(f"⚠️  มีแค่ {N_RUNS} ช่วงต่อเนื่อง — block bootstrap ต้องมีอย่างน้อย 5 ช่วง")    print("    CI ที่ได้จะแคบจนไร้ความหมาย (สุ่มช่วงคืนที่แล้วได้ชุดเดิมทุกรอบ)")    print("    skill horizon ที่คำนวณได้จึง **เชื่อไม่ได้** — อย่าเอาไปใส่เปเปอร์")    print("    ทางแก้: ตั้ง MIN_RUN ให้ต่ำลงในขั้น 3.1 เพื่อให้มีช่วงมากขึ้น\n")horizon, hrows = None, []for l in LEADS:    P, Q = ARR["pysteps"][l], ARR["persist"][l]    d = []    for _ in range(1000):        pick = rng.integers(0, len(idx_by_run), len(idx_by_run))        idx = np.concatenate([idx_by_run[k] for k in pick])        d.append(metrics(P[idx].sum(0))["CSI"] - metrics(Q[idx].sum(0))["CSI"])    lo, hi = np.percentile(d, [2.5, 97.5])    sig = lo > 0    hrows.append(dict(lead_min=l, skill=round(float(np.mean(d)), 4),                      lo=round(float(lo), 4), hi=round(float(hi), 4),                      ชนะ_persistence="ใช่" if sig else "ไม่"))    if sig:        horizon = lhdf = pd.DataFrame(hrows)if RELIABLE:    print(f"skill horizon = {horizon} นาที  (จาก {N_RUNS} ช่วงต่อเนื่อง)")else:    print(f"skill horizon = {horizon} นาที  ⚠️ เชื่อไม่ได้ — มีแค่ {N_RUNS} ช่วง")    horizon = Noneprint()hdf

### รูปสำหรับเปเปอร์

In [ ]:
import matplotlib.pyplot as pltfrom matplotlib import rcParamsrcParams.update({"font.family": "DejaVu Sans", "font.size": 9,                 "axes.linewidth": .8, "svg.fonttype": "none"})fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.2, 3.0), constrained_layout=True)L = tab["lead_min"]if cidf["block_lo"].notna().all():    ax1.fill_between(cidf["lead_min"], cidf["block_lo"], cidf["block_hi"],                     color="#1a5f66", alpha=.15, lw=0, label="95% CI (block bootstrap)")ax1.plot(L, tab["CSI_ps"], "o-", c="#1a5f66", lw=1.6, ms=4.5, label="CSI (optical flow)")ax1.plot(L, tab["CSI_pe"], "^:", c="#888", lw=1.3, ms=4.0, label="CSI (persistence)")if horizon:    ax1.axvline(horizon, c="#c25a1e", lw=1.0, ls="--")    ax1.annotate(f"skill horizon\n{horizon} min", xy=(horizon, ax1.get_ylim()[1]*.72),                 xytext=(4, 0), textcoords="offset points",                 fontsize=7.5, color="#c25a1e", va="center")ax1.set_xlabel("Lead time (min)"); ax1.set_ylabel("CSI")ax1.set_xticks(list(LEADS)); ax1.set_ylim(0, None)ax1.grid(alpha=.25, lw=.6); ax1.legend(frameon=False, fontsize=8)ax1.set_title("(a) Skill vs persistence", loc="left", fontsize=9.5, fontweight="bold")ax2.plot(L, tab["POD_ps"], "o-",  c="#1a5f66", lw=1.6, ms=4.5, label="POD")ax2.plot(L, tab["FAR_ps"], "s--", c="#c25a1e", lw=1.6, ms=4.0, label="FAR")ax2.plot(L, tab["BIAS_ps"], "d:", c="#666", lw=1.3, ms=4.0, label="BIAS")ax2.axhline(1.0, c="#bbb", lw=.7)ax2.set_xlabel("Lead time (min)"); ax2.set_ylabel("POD · FAR · BIAS")ax2.set_xticks(list(LEADS)); ax2.grid(alpha=.25, lw=.6)ax2.legend(frameon=False, fontsize=8)ax2.set_title("(b) Detection and false alarms", loc="left", fontsize=9.5, fontweight="bold")fig.savefig("/content/fig_skill.svg", bbox_inches="tight")fig.savefig("/content/fig_skill.png", dpi=300, bbox_inches="tight")plt.show()

### บันทึกผล + metadata`metadata` สำคัญพอ ๆ กับตัวเลข — ถ้า reviewer ขอ reproduce ต้องรู้ว่าตัวเลขนี้มาจาก commit ไหน คลังช่วงไหน กี่ origin engine อะไร threshold เท่าไรไม่งั้นอีกหกเดือนคุณเองก็ตอบไม่ได้

In [ ]:
tab.to_csv("/content/skill_table.csv", index=False)cidf.to_csv("/content/skill_ci.csv", index=False)hdf.to_csv("/content/skill_horizon.csv", index=False)pd.DataFrame(INFO).to_csv("/content/origins.csv", index=False)meta_out = dict(    commit=COMMIT, station=STATION, n_origins=N, n_runs=len(paths),    time_from=min(r["t0"] for r in INFO), time_to=max(r["t0"] for r in INFO),    engines=list(ENGINES), production_engine="pysteps",    threshold_dbz=THR, leads_min=list(LEADS), n_input_frames=N_IN,    despeckle=True, skill_horizon_min=horizon, n_continuous_runs=N_RUNS,    ci_reliable=bool(RELIABLE),    pysteps_version=md_.version("pysteps"),    runs=[p.name for p in paths],)Path("/content/backtest_meta.json").write_text(    json.dumps(meta_out, ensure_ascii=False, indent=2), encoding="utf-8")print(json.dumps(meta_out, ensure_ascii=False, indent=2))

In [ ]:
from google.colab import filesfor f in ("fig_skill.svg", "fig_skill.png", "skill_table.csv", "skill_ci.csv",          "skill_horizon.csv", "origins.csv", "backtest_meta.json"):    files.download(f"/content/{f}")

---## เทียบกับผลชุดเก่า| lead | CSI เก่า (light, ไม่ despeckle) | CSI ใหม่ ||---|---|---|| 15 | 0.465 | ดูตารางที่ 1 || 30 | 0.328 | || 60 | 0.199 | || 120 | 0.098 | |**ถ้าต่างกันไม่เกิน ~2%** → ปกติ เขียนในเปเปอร์ได้ว่า despeckle กับ optical flowให้ผลดีขึ้นเล็กน้อยแต่ไม่เปลี่ยนข้อสรุป**ถ้าต่างกันเกิน 5%** → หยุด อย่าเพิ่งใช้ ตรวจว่า- จำนวน origin ต่างจากเดิมมากไหม (คลังโตขึ้น ควรมากกว่า 502)- `despeckle` ตัดไปกี่เซลล์ (ถ้าหลักพันคือมากผิดปกติ อาจไปกินฝนจริง)- `BIAS` เพี้ยนไปจาก 1 มากไหมที่ lead สั้น**ถ้า CSI แย่ลง** → มีอะไรผิด ไม่ใช่ผลที่ควรได้ กลับมาบอกผมพร้อม `backtest_meta.json`